# 🔍 Azure AI Search — Lab AI-102

**Objectif**: Créer et interroger un index de recherche intelligent.

## Compétences AI-102 couvertes
- Concevoir un index avec types de champs
- Indexer des documents avec métadonnées
- Recherche full-text (BM25)
- Recherche sémantique (re-ranking neuronal)
- Filtres OData et facettes
- Highlighting et captions sémantiques
- Pipeline d'enrichissement AI (skillsets)

In [ ]:
%pip install azure-search-documents python-dotenv -q

In [ ]:
import os
from dotenv import load_dotenv
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex, SearchField, SearchFieldDataType,
    SimpleField, SearchableField,
    SemanticConfiguration, SemanticPrioritizedFields, SemanticField, SemanticSearch,
)
from azure.search.documents.models import QueryType, QueryCaptionType
from azure.core.credentials import AzureKeyCredential
from datetime import datetime, timezone

load_dotenv('../.env')

endpoint = os.getenv('AZURE_SEARCH_ENDPOINT')
key = os.getenv('AZURE_SEARCH_ADMIN_KEY')
index_name = 'lab-documents'

index_client = SearchIndexClient(endpoint=endpoint, credential=AzureKeyCredential(key))
search_client = SearchClient(endpoint=endpoint, index_name=index_name, credential=AzureKeyCredential(key))

print('✅ Clients Azure AI Search initialisés')

## 1. Créer un index de recherche

In [ ]:
# AI-102: Concevoir l'index avec les bons types de champs
# - SearchableField: full-text search avec analyzer
# - SimpleField: filtres exacts, tri, facettes
# - SemanticConfiguration: activer la recherche sémantique

fields = [
    SimpleField(name='id', type=SearchFieldDataType.String, key=True),
    SearchableField(name='title', type=SearchFieldDataType.String, analyzer_name='fr.lucene'),
    SearchableField(name='content', type=SearchFieldDataType.String, analyzer_name='fr.lucene'),
    SimpleField(name='doc_type', type=SearchFieldDataType.String, filterable=True, facetable=True),
    SimpleField(name='domain', type=SearchFieldDataType.String, filterable=True, facetable=True),
    SimpleField(name='language', type=SearchFieldDataType.String, filterable=True),
    SimpleField(name='date', type=SearchFieldDataType.DateTimeOffset, filterable=True, sortable=True),
    SimpleField(name='tags', type=SearchFieldDataType.Collection(SearchFieldDataType.String),
                filterable=True, facetable=True),
]

semantic_config = SemanticConfiguration(
    name='doc-semantic-config',
    prioritized_fields=SemanticPrioritizedFields(
        title_field=SemanticField(field_name='title'),
        content_fields=[SemanticField(field_name='content')],
        keywords_fields=[SemanticField(field_name='tags')],
    )
)

index = SearchIndex(
    name=index_name,
    fields=fields,
    semantic_search=SemanticSearch(configurations=[semantic_config])
)

result = index_client.create_or_update_index(index)
print(f"Index '{result.name}' créé/mis à jour")

## 2. Indexer des documents

In [ ]:
# AI-102: Les documents indexés doivent respecter le schéma de l'index

documents = [
    {
        'id': 'doc-001',
        'title': 'Contrat de prestation Azure AI Foundry',
        'content': 'Contrat de développement d\'une application de gestion documentaire basée sur Azure AI Foundry. Budget 85000 EUR. Durée 6 mois.',
        'doc_type': 'contrat',
        'domain': 'juridique',
        'language': 'fr',
        'date': datetime.now(timezone.utc).isoformat(),
        'tags': ['azure', 'ai', 'contrat', 'prestation'],
    },
    {
        'id': 'doc-002',
        'title': 'Rapport d\'analyse des coûts cloud 2024',
        'content': 'Analyse des coûts d\'infrastructure cloud pour l\'année 2024. Les services Azure AI représentent 35% du budget IT. Recommandations d\'optimisation incluses.',
        'doc_type': 'rapport',
        'domain': 'financier',
        'language': 'fr',
        'date': datetime.now(timezone.utc).isoformat(),
        'tags': ['cloud', 'coûts', 'azure', 'rapport', 'budget'],
    },
    {
        'id': 'doc-003',
        'title': 'Guide de déploiement Azure AI Language',
        'content': 'Guide technique pour déployer et configurer Azure AI Language Service. Inclut la configuration des endpoints, gestion des clés API, et optimisation des performances NLP.',
        'doc_type': 'guide',
        'domain': 'technique',
        'language': 'fr',
        'date': datetime.now(timezone.utc).isoformat(),
        'tags': ['azure', 'ai-language', 'déploiement', 'technique', 'nlp'],
    },
    {
        'id': 'doc-004',
        'title': 'Politique de sécurité et confidentialité des données',
        'content': 'Politique définissant les règles de sécurité pour le traitement des données personnelles. Conforme RGPD. Inclut la gestion des accès, chiffrement et audit.',
        'doc_type': 'politique',
        'domain': 'juridique',
        'language': 'fr',
        'date': datetime.now(timezone.utc).isoformat(),
        'tags': ['sécurité', 'rgpd', 'confidentialité', 'données', 'juridique'],
    },
]

results = search_client.upload_documents(documents=documents)
print(f"Documents indexés: {sum(1 for r in results if r.succeeded)}/{len(documents)}")
for r in results:
    status = '✅' if r.succeeded else '❌'
    print(f"  {status} {r.key} (HTTP {r.status_code})")

## 3. Recherche full-text

In [ ]:
# AI-102: Recherche full-text avec BM25 (TF-IDF amélioré)

import time
time.sleep(2)  # Attendre l'indexation

results = search_client.search(
    search_text='azure ai déploiement',
    top=5,
    highlight_fields='content,title',
    highlight_pre_tag='**',
    highlight_post_tag='**',
    include_total_count=True
)

print(f"Résultats pour 'azure ai déploiement': {results.get_count()}\n")
for r in results:
    print(f"Score: {r['@search.score']:.3f} | {r['title']}")
    if r.get('@search.highlights'):
        for field, highlights in r['@search.highlights'].items():
            print(f"  Highlight [{field}]: {highlights[0]}")
    print()

## 4. Recherche sémantique

In [ ]:
# AI-102: La recherche sémantique utilise un re-ranking neuronal
# Elle comprend l'intention et retourne les résultats les plus PERTINENTS
# (pas seulement les plus fréquents)

results = search_client.search(
    search_text='comment protéger les informations personnelles',
    query_type=QueryType.SEMANTIC,
    semantic_configuration_name='doc-semantic-config',
    query_caption=QueryCaptionType.EXTRACTIVE,
    top=3,
    include_total_count=True
)

print("Recherche sémantique: 'comment protéger les informations personnelles'\n")
for r in results:
    print(f"Score: {r['@search.score']:.3f} | {r['title']}")
    captions = r.get('@search.captions', [])
    if captions:
        print(f"  Caption sémantique: {captions[0].text}")
    print()

## 5. Filtres et facettes

In [ ]:
# AI-102: Les filtres OData permettent de restreindre les résultats
# Les facettes permettent le drill-down (navigation à facettes)

# Recherche filtrée par domaine
results = search_client.search(
    search_text='azure',
    filter="domain eq 'juridique'",
    facets=['doc_type', 'domain', 'tags,count:5'],
    top=10,
    include_total_count=True
)

print(f"Résultats filtrés (domain=juridique): {results.get_count()}")
for r in results:
    print(f"  • {r['title']} [{r['doc_type']}]")

print("\nFacettes disponibles:")
facets = results.get_facets()
if facets:
    for facet_name, facet_values in facets.items():
        print(f"  {facet_name}: {[(v['value'], v['count']) for v in facet_values[:3]]}")

## Résumé AI-102 — Azure AI Search

| Concept | Description |
|---------|------------|
| **Index** | Schéma définissant la structure des documents |
| **Indexer** | Composant qui charge les données dans l'index |
| **Skillset** | Pipeline d'enrichissement AI (OCR, NLP, embeddings) |
| **Full-text** | Recherche BM25 basée sur la fréquence des termes |
| **Sémantique** | Re-ranking neuronal pour la pertinence sémantique |
| **Vectorielle** | Recherche par similarité de vecteurs (embeddings) |
| **Filtre OData** | `domain eq 'juridique' and date gt 2024-01-01T00:00:00Z` |